# 第三級 · 訓練自己的金屬零件模型（Colab T4）

預設用 **Roboflow「Bolts」**（螺絲/螺帽/螺栓/墊圈，小、有標註、幾分鐘訓完）。
替代資料：**SteelDS** 輸送帶金屬（較大、Zenodo 慢；注意只有 a1/a2/a3 有標註，a4/a5 無標註）。

> 訓練需 GPU：執行階段 → 變更執行階段類型 → **T4 GPU**。

In [ ]:
# 1) 確認 GPU（訓練一定要）
import torch
assert torch.cuda.is_available(), '訓練需 GPU：執行階段→變更執行階段類型→T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) 安裝
!pip -q install ultralytics roboflow
import ultralytics; print(ultralytics.__version__)

## 取得資料集（方式 A 預設）

In [ ]:
# 方式 A（預設）：Roboflow Bolts —— 到 roboflow.com 註冊免費取得 API key
# 安全：用 getpass 執行時輸入，key 不會留在 notebook 裡（公開 repo 才安全）
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass('貼上你的 Roboflow API key（roboflow.com 免費取得）：'))
dataset = rf.workspace('bolts').project('bolts-final').version(1).download('yolov8')
DATA = dataset.location + '/data.yaml'
print('DATA =', DATA)

### （替代）方式 B：SteelDS 輸送帶金屬
`SUBSET='a1'`（**有標註**；a4/a5 無標註會訓練不出東西）。Zenodo 下載較慢。要用就解除下一格註解、改跑這格。

In [ ]:
# 方式 B（替代，預設不跑）：SteelDS a1（有標註）
# import os, glob, yaml
# SUBSET = 'a1'   # 用 a1/a2/a3（有標註）；勿用 a4/a5
# if not os.path.exists(SUBSET):
#     !wget -q --show-progress -O {SUBSET}.zip "https://zenodo.org/records/20271102/files/{SUBSET}.zip?download=1"
#     !unzip -q {SUBSET}.zip -d .
# root = os.path.dirname(os.path.dirname((glob.glob(f'**/{SUBSET}/train/images', recursive=True)+glob.glob('**/train/images', recursive=True))[0]))
# DATA = 'steelds.yaml'
# yaml.safe_dump({'path': os.path.abspath(root), 'train':'train/images', 'val':'test/images', 'nc':3, 'names':{0:'Background',1:'Steel',2:'Copper'}}, open(DATA,'w'))
# print('train 標註數:', len(glob.glob(root+'/train/labels/*.txt')), '| DATA =', DATA)

In [ ]:
# 3) Colab T4 微調（yolo11n）
from ultralytics import YOLO
m = YOLO('yolo11n.pt')
m.train(data=DATA, epochs=30, imgsz=640, batch=16, name='metal', exist_ok=True)
print('best 權重: runs/detect/metal/weights/best.pt')

In [ ]:
# 4) 驗證 mAP（應 > 0；若為 0 代表資料沒標註，換資料）
best = YOLO('runs/detect/metal/weights/best.pt')
mt = best.val(data=DATA)
print('mAP50:', round(mt.box.map50,3), '| mAP50-95:', round(mt.box.map,3))

In [ ]:
# 顯示訓練曲線 + 驗證偵測結果（執行後直接看到）
import os
from IPython.display import Image as IPImage, display
for f in ['results.png','confusion_matrix.png','val_batch0_pred.jpg']:
    pth = f'runs/detect/metal/{f}'
    if os.path.exists(pth): print(f); display(IPImage(pth, width=640))

In [ ]:
# 5) 在測試影像上看結果
import glob
from IPython.display import Image as IPImage, display
try: testdir = dataset.location + '/test/images'
except NameError: testdir = sorted(glob.glob('**/test/images', recursive=True))[0]
imgs = sorted(glob.glob(testdir + '/*'))[:6]
best.predict(imgs, save=True, project='result', name='metal', exist_ok=True, conf=0.3)
for p in sorted(glob.glob('result/metal/*'))[:3]: display(IPImage(p, width=520))

In [ ]:
# 6) 匯出 NCNN（給第二級 RPi 邊緣部署）
best.export(format='ncnn')
print('已匯出：runs/detect/metal/weights/best_ncnn_model')

In [ ]:
# 7) 把結果存到 Drive（給老師抓回上站）
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil
dst = '/content/drive/MyDrive/yolo-course/metal'; os.makedirs(dst + '/preds', exist_ok=True)
for p in glob.glob('runs/detect/metal/weights/best.pt'): shutil.copy(p, dst)
for p in glob.glob('runs/detect/metal/*.png') + glob.glob('runs/detect/metal/*.jpg'): shutil.copy(p, dst)
for p in glob.glob('result/metal/*'): shutil.copy(p, dst + '/preds/')
print('已複製到', dst, '| 檔數:', len(glob.glob(dst + '/**', recursive=True)))
drive.flush_and_unmount(); print('已 flush，可抓 yolo-course/metal/')